# Prétraitement des tweets SynTime

Transforme les fichiers `.tml` en un CSV avec, pour chaque tweet, le texte brut (entrée du modèle) et le texte annoté (référence TIMEX3).

In [ ]:
import pandas as pd
import re
from pathlib import Path
import xml.etree.ElementTree as ET

def extraire_texte_brut(texte_xml):
    """
    Extrait le texte brut en supprimant toutes les balises XML/TIMEX3
    """
    # Supprimer toutes les balises TIMEX3 mais garder le contenu
    texte_brut = re.sub(r'<TIMEX3[^>]*>', '', texte_xml)
    texte_brut = re.sub(r'</TIMEX3>', '', texte_brut)
    # Nettoyer les espaces multiples
    texte_brut = re.sub(r'\s+', ' ', texte_brut)
    return texte_brut.strip()

def extraire_tweets_depuis_tml(chemin_dossier):
    """
    Extrait tous les tweets des fichiers .tml et crée deux DataFrames
    
    Args:
        chemin_dossier: Chemin vers le dossier contenant les fichiers .tml
        
    Returns:
        tuple: (tweets_bruts_df, tweets_annotes_df)
    """
    
    tweets_bruts = []
    tweets_annotes = []
    
    # Parcourir tous les fichiers .tml
    fichiers_tml = list(Path(chemin_dossier).glob('**/*.tml'))
    
    print(f"Nombre de fichiers .tml trouvés: {len(fichiers_tml)}")
    
    for fichier in fichiers_tml:
        try:
            # Lire le fichier
            with open(fichier, 'r', encoding='utf-8') as f:
                contenu = f.read()
            
            # Parser le XML
            root = ET.fromstring(contenu)
            
            # Extraire le DOCID
            docid_elem = root.find('DOCID')
            docid = docid_elem.text if docid_elem is not None else fichier.name
            
            # Extraire la date de création (DCT)
            dct_elem = root.find('.//DCT/TIMEX3')
            dct = dct_elem.get('value') if dct_elem is not None else None
            
            # Extraire le texte
            text_elem = root.find('TEXT')
            if text_elem is not None:
                # Texte annoté (avec balises)
                texte_annote = ''.join(text_elem.itertext()).strip()
                
                # Reconstituer le texte avec balises pour garder la structure
                texte_annote_xml = ET.tostring(text_elem, encoding='unicode', method='xml')
                # Nettoyer la balise TEXT
                texte_annote_xml = re.sub(r'</?TEXT[^>]*>', '', texte_annote_xml)
                texte_annote_xml = texte_annote_xml.strip()
                
                # Texte brut (sans balises)
                texte_brut = extraire_texte_brut(texte_annote_xml)
                
                # Ajouter aux listes
                tweets_bruts.append({
                    'doc_id': docid,
                    'date_creation': dct,
                    'texte': texte_brut,
                    'fichier': fichier.name
                })
                
                tweets_annotes.append({
                    'doc_id': docid,
                    'date_creation': dct,
                    'texte_annote': texte_annote_xml,
                    'fichier': fichier.name
                })
                
        except Exception as e:
            print(f"Erreur lors du traitement de {fichier.name}: {str(e)}")
            continue
    
    # Créer les DataFrames
    tweets_bruts_df = pd.DataFrame(tweets_bruts)
    tweets_annotes_df = pd.DataFrame(tweets_annotes)
    
    return tweets_bruts_df, tweets_annotes_df


# Exemple d'utilisation
# Spécifier le chemin vers votre dossier contenant les fichiers .tml
chemin_dossier = "./tweets_dataset"  # À MODIFIER selon votre chemin

# Extraire les tweets
df_bruts, df_annotes = extraire_tweets_depuis_tml(chemin_dossier)

# Afficher les résultats
print(f"\n=== TWEETS BRUTS ===")
print(f"Nombre de tweets: {len(df_bruts)}")
print(df_bruts.head())

print(f"\n=== TWEETS ANNOTÉS ===")
print(f"Nombre de tweets: {len(df_annotes)}")
print(df_annotes.head())

# Sauvegarder en CSV
df_tweets = df_bruts
df_tweets["texte_annote"] = df_annotes["texte_annote"]
df_tweets.drop(columns=["doc_id","date_creation", "fichier"], inplace=True)
df_tweets["fichier"] = df_annotes["fichier"]

print("\n✓ Fichiers CSV sauvegardés: tweets_bruts.csv et tweets_annotes.csv")

# Exemple d'affichage d'un tweet
print("\n=== EXEMPLE DE TWEET ===")
if len(df_bruts) > 0:
    print(f"Brut: {df_bruts.iloc[0]['texte']}")
    print(f"Annoté: {df_annotes.iloc[0]['texte_annote']}")